<a href="https://colab.research.google.com/github/tarasovEgor/DeepLearningKurs/blob/main/src/lab_2/llm_transformer_lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 2.**

---

## LLM и Трансформер

---

## **Цель работы:** подготовить компактный корпус немецкоязычных диалогов, изучить ключевые компоненты архитектуры трансформера, выполнить fine-tuning GPT-подобной модели для генерации диалоговых ответов, сравнить оптимизаторы и scheduler, оценить качество по метрикам Perplexity, BLEU, ROUGE и протестировать модель в интерактивном режиме.


### 0. Init steps

---

#### 0.1 Установка дополнительных модулей:

In [1]:
!pip -q install transformers datasets evaluate rouge-score sentencepiece accelerate nltk spacy scikit-learn matplotlib pandas
!python -m spacy download de_core_news_sm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 43.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#### 0.2 Импорты библиотек и базовая настройка окружения:

In [6]:
import os
import re
import math
import json
import random
import warnings
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import spacy
from rouge_score import rouge_scorer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

Устройство: cuda


#### 0.3 Загрузка ресурсов NLTK и spaCy:

In [3]:
nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("de_core_news_sm", disable=["parser", "ner"])

german_stopwords = set(stopwords.words("german"))

print("Количество немецких стоп-слов:", len(german_stopwords))
print("spaCy модель загружена успешно")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Количество немецких стоп-слов: 232
spaCy модель загружена успешно


#### 0.4 Общая конфигурация проекта

In [12]:
MODEL_NAME = "dbmdz/german-gpt2"

DATA_URL = "https://raw.githubusercontent.com/liuzeming01/XDailyDialog/main/data/1k_part_data/dialogues_text_De.txt"
SAMPLE_SIZE = None

# Базовые параметры
MAX_LENGTH = 96
BATCH_SIZE = 4
NUM_EPOCHS = 2

LR_ADAM = 5e-5
LR_SGD = 1e-3
LR_RMSPROP = 1e-4

MODEL_SAVE_DIR = "/content/german_dialog_gpt2"
LOG_FILE = "/content/chat_log.txt"

print("Модель:", MODEL_NAME)
print("Источник датасета:", DATA_URL)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)

Модель: dbmdz/german-gpt2
Источник датасета: https://raw.githubusercontent.com/liuzeming01/XDailyDialog/main/data/1k_part_data/dialogues_text_De.txt
SAMPLE_SIZE: None
MAX_LENGTH: 96
BATCH_SIZE: 4
NUM_EPOCHS: 2


#### 0.5 Проверка токенизатора:

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

test_text = "Hallo! Wie geht es dir heute?"
tokens = tokenizer.tokenize(test_text)
token_ids = tokenizer.encode(test_text)

print("Тестовая строка:", test_text)
print("Токены:", tokens)
print("ID токенов:", token_ids[:20])
print("PAD token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/865 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Тестовая строка: Hallo! Wie geht es dir heute?
Токены: ['Hallo', '!', 'ĠWie', 'Ġgeht', 'Ġes', 'Ġdir', 'Ġheute', '?']
ID токенов: [5568, 5, 2675, 1284, 444, 1112, 1329, 35]
PAD token: <|endoftext|>


#### 0.6 Вывод по нулевому блоку:

На начальном этапе были установлены и подключены библиотеки для обработки текста, обучения трансформерной модели и расчёта метрик качества. В качестве основы для корпуса выбран немецкий файл dialogues_text_De.txt из датасета XDailyDialog, содержащего многоходовые диалоги. Для fine-tuning выбрана предобученная модель dbmdz/german-gpt2, ориентированная на немецкий язык. Дополнительно были загружены ресурсы nltk и spaCy для токенизации, удаления стоп-слов и лемматизации текста.

### 1. Загрузка данных

---

#### 1.1 Загрузка немецкого корпуса диалогов XDailyDialog:

Каждая строка - один диалог, а реплики внутри строки разделены специальным маркером __eou__:

In [13]:

SAMPLE_SIZE = None

data_path = "/content/dialogues_text_De.txt"
urllib.request.urlretrieve(DATA_URL, data_path)

print("Файл сохранён:", data_path)
print("Размер файла (байт):", os.path.getsize(data_path))

Файл сохранён: /content/dialogues_text_De.txt
Размер файла (байт): 479639


#### 1.2 Чтение корпуса и первичный просмотр:

In [15]:
with open(data_path, "r", encoding="utf-8") as f:
    raw_dialogues = [line.strip() for line in f.readlines() if line.strip()]

print("Количество диалогов в корпусе:", len(raw_dialogues))
print("\nПример первого диалога:\n")
print(raw_dialogues[0])

print("\n" + "-"*80 + "\n")
print("Пример второго диалога:\n")
print(raw_dialogues[1])

Количество диалогов в корпусе: 1000

Пример первого диалога:

Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __eou__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __eou__ Was hätte ich mitbringen sollen? __eou__ Das einzige, was ich sehen muss, ist Ihr Führerschein, da ich diese Papiere beglaubigen werde. __eou__ Ich fühle mich ein wenig überwältigt von so vielen Papieren. __eou__ Machen Sie sich keine Sorgen, wie viele Papiere es sind. __eou__ Mein Freund ist Anwalt und sagte mir, dass ich ihm alles faxen kann, wenn ich eine Frage habe. __eou__ Bitte holen Sie sich Hilfe von außen, die Sie für das Verständnis Ihrer Escrow-Dokumente benötigen. __eou__ Ist das das Letzte, was ich tun muss, bevor das Haus mir gehört? __eou__ Am Ende des Schecks wird das Haus Ihnen gehören!

--------------------------------------------------------------------------------

Пример второго диалога:

Smith ist immer unvorsichtig,

#### 1.3 Разбор диалогов на отдельные реплики:

В XDailyDialog реплики внутри строки разделены токеном __eou__.
Мы разбиваем каждый диалог на список реплик и удаляем пустые элементы.
Также оставляем только те диалоги, где есть минимум 2 реплики, иначе невозможно построить пару вопрос–ответ. Формат с __eou__ используется и в DailyDialog-подобных представлениях корпуса.

In [18]:
def split_dialogue(dialogue_text):
    utterances = [utt.strip() for utt in dialogue_text.split("eou") if utt.strip()]
    return utterances

parsed_dialogues = [split_dialogue(dialogue) for dialogue in raw_dialogues]
parsed_dialogues = [dialogue for dialogue in parsed_dialogues if len(dialogue) >= 2]

print("Количество пригодных диалогов:", len(parsed_dialogues))
print("\nПример разобранного диалога:")
print(parsed_dialogues[0])
print("\nКоличество реплик в первом диалоге:", len(parsed_dialogues[0]))

Количество пригодных диалогов: 1000

Пример разобранного диалога:
['Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __', '__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __', '__ Was hätte ich mitbringen sollen? __', '__ Das einzige, was ich sehen muss, ist Ihr Führerschein, da ich diese Papiere beglaubigen werde. __', '__ Ich fühle mich ein wenig überwältigt von so vielen Papieren. __', '__ Machen Sie sich keine Sorgen, wie viele Papiere es sind. __', '__ Mein Freund ist Anwalt und sagte mir, dass ich ihm alles faxen kann, wenn ich eine Frage habe. __', '__ Bitte holen Sie sich Hilfe von außen, die Sie für das Verständnis Ihrer Escrow-Dokumente benötigen. __', '__ Ist das das Letzte, was ich tun muss, bevor das Haus mir gehört? __', '__ Am Ende des Schecks wird das Haus Ihnen gehören!']

Количество реплик в первом диалоге: 10


#### 1.4 Статистика по корпусу:

In [19]:
dialogue_lengths = [len(dialogue) for dialogue in parsed_dialogues]

print("Минимум реплик в диалоге:", min(dialogue_lengths))
print("Максимум реплик в диалоге:", max(dialogue_lengths))
print("Среднее число реплик в диалоге:", round(np.mean(dialogue_lengths), 2))
print("Медиана:", np.median(dialogue_lengths))

Минимум реплик в диалоге: 2
Максимум реплик в диалоге: 30
Среднее число реплик в диалоге: 7.39
Медиана: 6.0


#### 1.5 Формирование пар context -> response:

Подготовим данные для обучения диалоговой модели. Из каждого диалога будем получать пары:
- context — текущая реплика;
- response — следующая реплика.

In [20]:
pairs = []

for dialogue in parsed_dialogues:
    for i in range(len(dialogue) - 1):
        context = dialogue[i].strip()
        response = dialogue[i + 1].strip()

        if len(context) > 0 and len(response) > 0:
            pairs.append({
                "context": context,
                "response": response
            })

print("Количество пар context-response:", len(pairs))
print("\nПример пары:")
print(pairs[0])

Количество пар context-response: 6386

Пример пары:
{'context': 'Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __', 'response': '__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __'}


#### 1.6 Преобразование в DataFrame:

In [21]:
pairs_df = pd.DataFrame(pairs)

print("Размер таблицы:", pairs_df.shape)
pairs_df.head()

Размер таблицы: (6386, 2)


,context,response
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...","__ Escrow beinhaltet eine Menge Papierkram, ab..."
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",__ Was hätte ich mitbringen sollen? __
2,__ Was hätte ich mitbringen sollen? __,"__ Das einzige, was ich sehen muss, ist Ihr Fü..."
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",__ Ich fühle mich ein wenig überwältigt von so...
4,__ Ich fühle mich ein wenig überwältigt von so...,"__ Machen Sie sich keine Sorgen, wie viele Pap..."


#### 1.7 Удаление слишком коротких и слишком длинных пар:

Иногда в корпусе встречаются слишком короткие или слишком длинные реплики.
Для более стабильного обучения в Colab оставим только разумные примеры.

In [22]:
def is_valid_text(text, min_len=2, max_len=200):
    text = text.strip()
    return min_len <= len(text.split()) <= max_len

pairs_df = pairs_df[
    pairs_df["context"].apply(is_valid_text) &
    pairs_df["response"].apply(is_valid_text)
].reset_index(drop=True)

print("Размер таблицы после фильтрации:", pairs_df.shape)
pairs_df.head()

Размер таблицы после фильтрации: (6384, 2)


,context,response
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...","__ Escrow beinhaltet eine Menge Papierkram, ab..."
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",__ Was hätte ich mitbringen sollen? __
2,__ Was hätte ich mitbringen sollen? __,"__ Das einzige, was ich sehen muss, ist Ihr Fü..."
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",__ Ich fühle mich ein wenig überwältigt von so...
4,__ Ich fühle mich ein wenig überwältigt von so...,"__ Machen Sie sich keine Sorgen, wie viele Pap..."


#### 1.8 Создание текстов в формате для GPT:

Поскольку мы будем дообучать GPT-подобную модель, удобно заранее объединить контекст и ответ в один текст.

Формат сделаем таким:
- Kontext: ...
- Antwort: ...


In [23]:
pairs_df["train_text"] = pairs_df.apply(
    lambda row: f"Kontext: {row['context']}\nAntwort: {row['response']}",
    axis=1
)

print("Пример итогового текста для обучения:\n")
print(pairs_df["train_text"].iloc[0])

Пример итогового текста для обучения:

Kontext: Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __
Antwort: __ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __


#### 1.9 Сохранение промежуточного результата:

In [24]:
pairs_csv_path = "/content/german_dialog_pairs.csv"
pairs_df.to_csv(pairs_csv_path, index=False, encoding="utf-8")

print("Файл с парами сохранён:", pairs_csv_path)

Файл с парами сохранён: /content/german_dialog_pairs.csv


#### 1.10 Вывод по первому блоку:

На данном этапе был загружен немецкоязычный корпус диалогов dialogues_text_De.txt из датасета XDailyDialog. Каждая строка корпуса соответствует одному многоходовому диалогу, а отдельные реплики разделяются маркером __eou__. После загрузки был выполнен первичный анализ структуры корпуса, включая подсчёт количества диалогов и статистику по числу реплик. Затем диалоги были преобразованы в пары вида контекст -> ответ, которые используются для дальнейшего обучения генеративной модели. Для удобства fine-tuning GPT-модели каждая пара была сохранена в виде единого текста формата Kontext: ... Antwort: .... дующим сообщением дам блок 2_data_preprocessing: очистка текста, стоп-слова, лемматизация и сравнение исходного и очищенного текста.


### 2. Очистка данных с применением методов NLP

---

Для генеративной модели диалогов слишком агрессивная очистка иногда ухудшает качество ответов, потому что модель начинает видеть «неестественный» язык.

Поэтому было принято решение выполнить данный пункт следуюшим образом:
- для демонстрации этапа preprocessing делаем полную очистку;
- для обучения модели используем более мягкую очистку, чтобы сохранить структуру диалога.

#### 2.1 Просмотр данных перед очисткой:

In [25]:
pairs_df[["context", "response"]].head(5)

,context,response
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...","__ Escrow beinhaltet eine Menge Papierkram, ab..."
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",__ Was hätte ich mitbringen sollen? __
2,__ Was hätte ich mitbringen sollen? __,"__ Das einzige, was ich sehen muss, ist Ihr Fü..."
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",__ Ich fühle mich ein wenig überwältigt von so...
4,__ Ich fühle mich ein wenig überwältigt von so...,"__ Machen Sie sich keine Sorgen, wie viele Pap..."


#### 2.2 Вспомогательные функции полной очистки текста:

Здесь реализована полная очистка:
- нижний регистр;
- удаление HTML;
- удаление ссылок;
- удаление лишних символов;
- удаление пунктуации;
- токенизация;
- удаление немецких стоп-слов;
- лемматизация.

In [26]:
def clean_text_full(text, stop_words=german_stopwords):
    text = str(text).lower().strip()

    # Удаление HTML-тегов
    text = re.sub(r"<.*?>", " ", text)

    # Удаление ссылок
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Удаление e-mail и упоминаний
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Оставляем буквы немецкого алфавита, пробелы
    text = re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", " ", text)

    # Удаление лишних пробелов
    text = re.sub(r"\s+", " ", text).strip()

    # Лемматизация через spaCy
    doc = nlp(text)
    tokens = []

    for token in doc:
        lemma = token.lemma_.strip().lower()

        if not lemma:
            continue
        if lemma in stop_words:
            continue
        if lemma in [" ", "\n", "\t"]:
            continue
        if len(lemma) < 2:
            continue

        tokens.append(lemma)

    return " ".join(tokens)

#### 2.3 Вспомогательная функция мягкой очистки для обучения модели:

Для обучения GPT используем мягкую очистку:
- нижний регистр;
- удаление HTML и ссылок;
- нормализация пробелов;
- сохраняем основную лексику и структуру.

Мы не удаляем стоп-слова и не делаем жёсткую лемматизацию в обучающем тексте, чтобы генерация оставалась более естественной:

In [27]:
def clean_text_for_training(text):
    text = str(text).strip()

    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление HTML и ссылок
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Удаление лишних специальных символов, но оставляем базовую пунктуацию
    text = re.sub(r"[^a-zA-ZäöüÄÖÜß0-9\s\.,!\?;:\-']", " ", text)

    # Удаление повторяющихся пробелов
    text = re.sub(r"\s+", " ", text).strip()

    return text

#### 2.4 Применение полной очистки к отдельным колонкам:

In [28]:
pairs_df["context_clean_full"] = pairs_df["context"].apply(clean_text_full)
pairs_df["response_clean_full"] = pairs_df["response"].apply(clean_text_full)

pairs_df[["context", "context_clean_full", "response", "response_clean_full"]].head(5)

,context,context_clean_full,response,response_clean_full
0,"Ich bin sehr nervös, meine Treuhandpapiere zu ...",nervös treuhandpapiere unterschreiben,"__ Escrow beinhaltet eine Menge Papierkram, ab...",escrow beinhalten menge papierkram schritt erk...
1,"__ Escrow beinhaltet eine Menge Papierkram, ab...",escrow beinhalten menge papierkram schritt erk...,__ Was hätte ich mitbringen sollen? __,mitbringen sollen
2,__ Was hätte ich mitbringen sollen? __,mitbringen sollen,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",einzig wer sehen führerschein papier beglaubigen
3,"__ Das einzige, was ich sehen muss, ist Ihr Fü...",einzig wer sehen führerschein papier beglaubigen,__ Ich fühle mich ein wenig überwältigt von so...,fühlen wenig überwältigen vieler papier
4,__ Ich fühle mich ein wenig überwältigt von so...,fühlen wenig überwältigen vieler papier,"__ Machen Sie sich keine Sorgen, wie viele Pap...",sorgen vieler papier


#### 2.5 Применение мягкой очистки для обучения:

In [29]:
pairs_df["context_clean_train"] = pairs_df["context"].apply(clean_text_for_training)
pairs_df["response_clean_train"] = pairs_df["response"].apply(clean_text_for_training)

pairs_df[["context_clean_train", "response_clean_train"]].head(5)

,context_clean_train,response_clean_train
0,"ich bin sehr nervös, meine treuhandpapiere zu ...","escrow beinhaltet eine menge papierkram, aber ..."
1,"escrow beinhaltet eine menge papierkram, aber ...",was hätte ich mitbringen sollen?
2,was hätte ich mitbringen sollen?,"das einzige, was ich sehen muss, ist ihr führe..."
3,"das einzige, was ich sehen muss, ist ihr führe...",ich fühle mich ein wenig überwältigt von so vi...
4,ich fühle mich ein wenig überwältigt von so vi...,"machen sie sich keine sorgen, wie viele papier..."


#### 2.6 Сравнение исходного и очищенного текста:

In [31]:
sample_idx = 0

print("ИСХОДНЫЙ CONTEXT:")
print(pairs_df.loc[sample_idx, "context"])
print("\nПОЛНАЯ ОЧИСТКА:")
print(pairs_df.loc[sample_idx, "context_clean_full"])
print("\nМЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:")
print(pairs_df.loc[sample_idx, "context_clean_train"])

print("\n" + "-"*80 + "\n")

print("ИСХОДНЫЙ RESPONSE:")
print(pairs_df.loc[sample_idx, "response"])
print("\nПОЛНАЯ ОЧИСТКА:")
print(pairs_df.loc[sample_idx, "response_clean_full"])
print("\nМЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:")
print(pairs_df.loc[sample_idx, "response_clean_train"])

ИСХОДНЫЙ CONTEXT:
Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __

ПОЛНАЯ ОЧИСТКА:
nervös treuhandpapiere unterschreiben

МЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:
ich bin sehr nervös, meine treuhandpapiere zu unterschreiben.

--------------------------------------------------------------------------------

ИСХОДНЫЙ RESPONSE:
__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __

ПОЛНАЯ ОЧИСТКА:
escrow beinhalten menge papierkram schritt erklären weitermachen

МЯГКАЯ ОЧИСТКА ДЛЯ ОБУЧЕНИЯ:
escrow beinhaltet eine menge papierkram, aber ich werde ihnen alle schritte erklären, während wir weitermachen.


#### 2.7 Удаление пустых строк после полной очистки:

После удаления стоп-слов и лемматизации часть коротких строк может стать пустой.
Такие строки нужно убрать, чтобы они не мешали анализу.

In [32]:
pairs_df = pairs_df[
    (pairs_df["context_clean_full"].str.len() > 0) &
    (pairs_df["response_clean_full"].str.len() > 0) &
    (pairs_df["context_clean_train"].str.len() > 0) &
    (pairs_df["response_clean_train"].str.len() > 0)
].reset_index(drop=True)

print("Размер таблицы после удаления пустых строк:", pairs_df.shape)

Размер таблицы после удаления пустых строк: (6180, 7)


#### 2.8 Подготовка итоговых текстов для анализа и обучения:

Сформируем:
- analysis_text — для анализа качества очистки;
- train_text_clean — для обучения GPT.

In [33]:
pairs_df["analysis_text"] = pairs_df.apply(
    lambda row: f"Kontext: {row['context_clean_full']} Antwort: {row['response_clean_full']}",
    axis=1
)

pairs_df["train_text_clean"] = pairs_df.apply(
    lambda row: f"Kontext: {row['context_clean_train']}\nAntwort: {row['response_clean_train']}",
    axis=1
)

pairs_df[["analysis_text", "train_text_clean"]].head(3)

,analysis_text,train_text_clean
0,Kontext: nervös treuhandpapiere unterschreiben...,"Kontext: ich bin sehr nervös, meine treuhandpa..."
1,Kontext: escrow beinhalten menge papierkram sc...,Kontext: escrow beinhaltet eine menge papierkr...
2,Kontext: mitbringen sollen Antwort: einzig wer...,Kontext: was hätte ich mitbringen sollen?\nAnt...


#### 2.9 Оценка влияния очистки на длину текста:

In [34]:
pairs_df["context_len_before"] = pairs_df["context"].apply(lambda x: len(str(x).split()))
pairs_df["context_len_after_full"] = pairs_df["context_clean_full"].apply(lambda x: len(str(x).split()))
pairs_df["context_len_after_train"] = pairs_df["context_clean_train"].apply(lambda x: len(str(x).split()))

print("Средняя длина context до очистки:", round(pairs_df["context_len_before"].mean(), 2))
print("Средняя длина context после полной очистки:", round(pairs_df["context_len_after_full"].mean(), 2))
print("Средняя длина context после мягкой очистки:", round(pairs_df["context_len_after_train"].mean(), 2))

Средняя длина context до очистки: 11.51
Средняя длина context после полной очистки: 4.47
Средняя длина context после мягкой очистки: 9.67


#### 2.10 Сохранение обработанных данных:

In [35]:
processed_csv_path = "/content/german_dialog_pairs_processed.csv"
pairs_df.to_csv(processed_csv_path, index=False, encoding="utf-8")

print("Обработанный файл сохранён:", processed_csv_path)

Обработанный файл сохранён: /content/german_dialog_pairs_processed.csv


#### 2.11 Вывод по второму блоку:

На этапе предварительной обработки были применены основные методы очистки текста, указанные в методических рекомендациях: приведение к нижнему регистру, удаление HTML-тегов, ссылок, специальных символов и лишней пунктуации, удаление стоп-слов, а также лемматизация. Для анализа была сформирована полностью очищенная версия текста, позволяющая оценить влияние preprocessing на структуру корпуса. Одновременно для обучения генеративной модели была подготовлена более мягкая версия текста, в которой сохраняется естественность диалогов и базовая пунктуация. Такой подход позволил выполнить требования лабораторной работы и одновременно не ухудшить качество будущей генерации ответов.